# CORDEX-ML-Bench train-val split comparison

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
desc = ""
dataset_name="ALPS_domain-Emulator_hist_future-CNRMCM5-perfect"
target_vars = ["tasmax", "pr"]

In [ ]:
import cartopy.feature as cfeature
import functools
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import xarray as xr

from mlde_utils import platecarree
from mlde_analysis import plot_map
from mlde_analysis.display import pretty_table
from mlde_analysis.data import open_dataset_split
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import mean_bias, std_bias, stat_bias, plot_freq_density, plot_distribution_figure, compute_metrics, DIST_THRESHOLDS, plot_freq_density_figure


In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
train_ds = open_dataset_split(dataset_name, split="train").expand_dims(model=["RCM train"]).expand_dims(sample_id=[0])
val_ds = open_dataset_split(dataset_name, split="val")

MODELLABEL2SPEC = {"RCM train": {"color": "red", "order": 1}}

ATTRS = {
    "pr": {
        "units": "mm/day",
        "standard_name": "precipitation_flux",
        "long_name": f"Precip.",
    },
    "tasmax": {
        "units": "K",
        "standard_name": "air_temperature",
        "long_name": "Tas Max",
    },
}

for var in target_vars:
    train_ds[var].attrs.update(ATTRS[var])
    val_ds[var].attrs.update(ATTRS[var])

In [ ]:
train_ds

In [ ]:
val_ds

In [ ]:
for var in target_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    
    hist_das = train_ds[var]
    target_da = val_ds[var]
    
    metrics_ds = compute_metrics(hist_das, target_da, thresholds=DIST_THRESHOLDS[var])

    pretty_table(metrics_ds, round=4)
    
    normalize=(var == "pr")
    mean_biases = train_ds[var].groupby("model").map(mean_bias, target_da=target_da, normalize=(var=="pr"))

    std_biases = train_ds[var].groupby("model").map(std_bias, target_da=target_da, normalize=(var=="pr"))
    f_q999_bias = functools.partial(stat_bias, stat_func=functools.partial(xr.DataArray.quantile, q=0.999))
    q999_biases = train_ds[var].groupby("model").map(f_q999_bias, target_da=target_da, normalize=(var=="pr"))
    
    bias_kwargs = {"style": f"{var}Bias"}
    for fd_kwargs in [{"yscale": "log", "target_label": "RCM val"}, {"yscale": "linear", "target_label": "RCM val"}]:
        fig = plt.figure(layout="constrained", figsize=(5.5, 7))
        error_fig = plt.figure(layout="constrained", figsize=(5.5, 2.5))
        error_axd = error_fig.subplot_mosaic([["Error"]])
        error_ax = error_axd["Error"]
        axd = plot_distribution_figure(
            fig,
            hist_das,
            target_da,
            {"meanb": mean_biases, "stdb": std_biases, "q999b": q999_biases,},
            MODELLABEL2SPEC,
            error_ax=error_ax, hrange=VAR_RANGES[var], fd_kwargs=fd_kwargs, bias_kwargs=bias_kwargs
        )
        if var == "relhum150cm":
            axd["Density"].axvline(x=100, color='k', linestyle='--', linewidth=1)
        
        plt.show()

In [ ]:
for var in target_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
        
    metrics_ds = xr.concat([ compute_metrics(
        season_da, val_ds[var].where(val_ds.time.dt.season==season, drop=True), thresholds=DIST_THRESHOLDS[var]
    ).expand_dims(season=[season]) for season, season_da in train_ds[var].groupby("time.season") ], dim="season")
        
    pretty_table(metrics_ds, round=4, dim_order=["season", "model"])

    for season, season_da in train_ds[var].groupby("time.season"):
        IPython.display.display_markdown(f"#### {season}", raw=True)
        hist_das = season_da
        target_da = val_ds[var].where(val_ds.time.dt.season==season, drop=True)

        normalize=(var == "pr")
        mean_biases = season_da.groupby("model").map(mean_bias, target_da=target_da, normalize=normalize)
        std_biases = season_da.groupby("model").map(std_bias, target_da=target_da, normalize=normalize)
        
        f_q999_bias = functools.partial(stat_bias, stat_func=functools.partial(xr.DataArray.quantile, q=0.999))
        q999_biases = season_da.groupby("model").map(f_q999_bias, target_da=target_da, normalize=normalize)
        
        bias_kwargs = {"style": f"{var}Bias"}
        for fd_kwargs in [{"yscale": "log"}, {"yscale": "linear"}]:
            if var == "pr" and fd_kwargs["yscale"] == "linear":
                continue
            fig = plt.figure(layout="constrained", figsize=(5.5, 6.5))
            axd = plot_distribution_figure(
                fig,
                hist_das,
                target_da,
                {"meanb": mean_biases, "stdb": std_biases, "q999b": q999_biases,},
                MODELLABEL2SPEC,
                hrange=VAR_RANGES[var],
                fd_kwargs=fd_kwargs,
                bias_kwargs=bias_kwargs,
            )
            if var == "relhum150cm":
                axd["Density"].axvline(x=100, color='k', linestyle='--', linewidth=1)
            
        plt.show()